# `local_embedding` — LocalDynamicEmbedding

Módulo para gerar *embeddings* de texto localmente com uma **API fluente** (encadeável), troca dinâmica de provedor e recuperação por similaridade.

O fluxo típico é: **configurar → processar texto → acessar chunks → recuperar por similaridade → exportar**.

---

## Como executar

O módulo usa *imports* relativos (`EmbeddingFactory`), então precisa ser rodado como módulo, com a flag `-m`:

```bash
python -m src.embedding.modules.local_embedding
```

Rodar o arquivo direto (`python arquivo.py`) quebra por causa do import relativo.

---

## Início rápido

```python
from src.embedding.modules.local_embedding.module import (
    LocalDynamicEmbedding,
    EmbeddingFactory,
)

# Monta o pipeline (encadeando as configurações)
pipeline = (
    LocalDynamicEmbedding()
    .with_fake_embeddings(size=384)          # embeddings falsos p/ testes
    .with_splitter(chunk_size=120, chunk_overlap=20)
    .with_top_k(3)
)

# Processa um texto e recebe a quantidade de chunks gerados
qtd = pipeline.process_text("seu texto...", metadata={"fonte": "exemplo"})

# Recupera os trechos mais similares a uma consulta
resultados = pipeline.retrieve("sua pergunta?", include_embedding=True)
```

---

## Provedores disponíveis

```python
EmbeddingFactory.available()   # -> lista os provedores registrados
```

Use para descobrir quais backends de embedding estão instalados/registrados no ambiente.

---

## Construindo o pipeline (API fluente)

Cada método `with_*` retorna a própria instância, permitindo encadeamento. Se você **não** definir um provedor de embeddings, a classe cai no `fake` automaticamente.

| Método | Efeito | Parâmetros |
|---|---|---|
| `.with_fake_embeddings(size=384)` | Usa embeddings falsos (vetores de dimensão `size`) — ideal para testes sem dependências externas | `size`: dimensão do vetor |
| `.with_splitter(chunk_size, chunk_overlap)` | Configura como o texto é dividido em chunks | `chunk_size`: tamanho do chunk; `chunk_overlap`: sobreposição entre chunks |
| `.with_top_k(k)` | Define o número padrão de resultados retornados na recuperação | `k`: quantidade de resultados |

### Construtores de fábrica (troca de provedor em 1 linha)

Para usar um provedor real, existem construtores alternativos, por exemplo:

```python
pipeline_openai = LocalDynamicEmbedding.from_openai_embeddings(
    model="text-embedding-3-large",
    chunk_size=3000,
    chunk_overlap=300,
    top_k=5,
)
```

> Requer as dependências do provedor (ex.: OpenAI) instaladas e configuradas.

---

## Processando texto

```python
qtd = pipeline.process_text(texto, metadata={"fonte": "apostila_energia"})
```

- **Retorno:** número de chunks gerados **nesta chamada** (`int`).
- **`metadata`** (opcional): dicionário anexado a cada chunk gerado a partir desse texto.
- Pode ser chamado várias vezes; os chunks se acumulam.

```python
pipeline.total_chunks   # total acumulado de chunks em todas as chamadas
```

---

## Acessando os chunks

`pipeline.chunks` é a lista de objetos chunk. Cada chunk expõe:

| Atributo | Descrição |
|---|---|
| `chunk.index` | Índice do chunk |
| `chunk.content` | Texto do chunk |
| `chunk.length` | Comprimento do texto |
| `chunk.dim` | Dimensão do vetor de embedding |
| `chunk.metadata` | Metadados associados |
| `chunk.embedding` | Vetor de embedding (lista de floats) |

Exemplo:

```python
for chunk in pipeline.chunks:
    preview = chunk.content[:60].replace("\n", " ")
    print(f"[{chunk.index}] len={chunk.length} dim={chunk.dim} meta={chunk.metadata}")
    print(f"     texto: {preview}...")
    print(f"     embedding[:3]: {chunk.embedding[:3]}")
```

---

## Recuperação por similaridade

```python
resultados = pipeline.retrieve("como gerar eletricidade a partir do vento?",
                               include_embedding=True)
```

- **Retorno:** lista de dicionários ordenados por relevância. Cada item contém:
  - `score` — pontuação de similaridade (`float`)
  - `content` — texto do trecho recuperado
  - `embedding` — vetor do trecho (somente se `include_embedding=True`)
- A quantidade de resultados segue o `top_k` configurado.

```python
for i, r in enumerate(pipeline.retrieve(consulta, include_embedding=True), 1):
    print(f"#{i} score={r['score']:.4f} | dim={len(r['embedding'])}")
    print(f"   {r['content'][:70]}...")
```

---

## Exportando os chunks (pronto para JSON)

```python
dados = pipeline.get_chunks(include_embedding=False)
```

- **Retorno:** lista de dicionários serializáveis (ideal para `json.dumps`).
- Use `include_embedding=False` para omitir os vetores e gerar uma saída enxuta.

```python
import json
exemplo = pipeline.get_chunks(include_embedding=False)[0]
print(json.dumps(exemplo, ensure_ascii=False, indent=2))
```

---

## Exemplo completo

```python
from src.embedding.modules.local_embedding.module import (
    LocalDynamicEmbedding, EmbeddingFactory,
)
import json

print("Provedores disponíveis:", EmbeddingFactory.available())

texto = (
    "A energia solar é uma fonte renovável...\n\n"
    "A energia eólica aproveita a força dos ventos...\n\n"
    "Já os combustíveis fósseis são fontes não renováveis...\n\n"
    "O uso de baterias é essencial para armazenar energia..."
)

# 1) Configuração (sem embeddings -> usa fake automaticamente)
pipeline = (
    LocalDynamicEmbedding()
    .with_fake_embeddings(size=384)
    .with_splitter(chunk_size=120, chunk_overlap=20)
    .with_top_k(3)
)

# 2) Processamento
qtd = pipeline.process_text(texto, metadata={"fonte": "apostila_energia"})
print(f"Chunks gerados: {qtd} | total: {pipeline.total_chunks}")

# 3) Inspeção dos chunks
for chunk in pipeline.chunks:
    print(f"[{chunk.index}] dim={chunk.dim} meta={chunk.metadata}")

# 4) Recuperação
for i, r in enumerate(pipeline.retrieve("energia do vento?", include_embedding=True), 1):
    print(f"#{i} score={r['score']:.4f} -> {r['content'][:60]}...")

# 5) Exportação JSON
print(json.dumps(pipeline.get_chunks(include_embedding=False)[0],
                 ensure_ascii=False, indent=2))
```

---

## Referência rápida da API

**`EmbeddingFactory`**
- `EmbeddingFactory.available()` → lista de provedores

**`LocalDynamicEmbedding`**
- `LocalDynamicEmbedding()` → instância vazia (fallback: fake)
- `.with_fake_embeddings(size=384)` → *self*
- `.with_splitter(chunk_size, chunk_overlap)` → *self*
- `.with_top_k(k)` → *self*
- `.from_openai_embeddings(model, chunk_size, chunk_overlap, top_k)` → nova instância (classmethod)
- `.process_text(texto, metadata=None)` → `int` (chunks gerados)
- `.total_chunks` → `int` (acumulado)
- `.chunks` → lista de objetos chunk (`.index`, `.content`, `.length`, `.dim`, `.metadata`, `.embedding`)
- `.retrieve(consulta, include_embedding=False)` → lista de dicts (`score`, `content`, [`embedding`])
- `.get_chunks(include_embedding=False)` → lista de dicts serializáveis

---

## Observações

- Sem provedor configurado, o pipeline usa **fake embeddings** — perfeito para testar o fluxo sem depender de APIs externas.
- Trocar de provedor é uma única linha (via construtor de fábrica), sem mudar o restante do código.
- Sempre execute como módulo (`python -m ...`) por causa dos imports relativos.

---

> Esta documentação descreve a **API pública observável** a partir do arquivo de demonstração. Detalhes internos de implementação (algoritmo de similaridade, formato exato do splitter, provedores registrados) dependem do código-fonte do módulo. Se quiser, me envie `module.py` e eu complemento com esses detalhes.
